# Entrega 2: Ingeniería de Características
### Detección de Tráfico Malicioso en Redes IoT
**Santiago Vieira Ceballos — Sara Franco Taborda — Sara Medina Molina**

## 1. Carga y comprensión de los datos

El dataset utilizado es el *Network IoT Dataset* (NICA), disponible en Kaggle,
el cual contiene registros de flujos de red con características extraídas de
múltiples capas y protocolos (DNS, HTTP, SSL), etiquetados como tráfico normal
o como alguno de los tipos de ciberataques presentes en el conjunto.

A continuación se carga el archivo CSV desde el directorio de datos del proyecto.
Los espacios en blanco se tratan como valores nulos (`na_values=[' ']`) para
garantizar una correcta identificación de datos faltantes desde el inicio.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path().resolve().parent / "Data"
data_file = "train_test_network.csv"
data_path = DATA_DIR / data_file

df = pd.read_csv(data_path, na_values=[' '])
df.head()

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


Se inspecciona el número de variables, registros y el tipo de dato asignado a cada columna.

In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211043 entries, 0 to 211042
Data columns (total 44 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   src_ip                  211043 non-null  object 
 1   src_port                211043 non-null  int64  
 2   dst_ip                  211043 non-null  object 
 3   dst_port                211043 non-null  int64  
 4   proto                   211043 non-null  object 
 5   service                 211043 non-null  object 
 6   duration                211043 non-null  float64
 7   src_bytes               211043 non-null  int64  
 8   dst_bytes               211043 non-null  int64  
 9   conn_state              211043 non-null  object 
 10  missed_bytes            211043 non-null  int64  
 11  src_pkts                211043 non-null  int64  
 12  src_ip_bytes            211043 non-null  int64  
 13  dst_pkts                211043 non-null  int64  
 14  dst_ip_bytes        

In [2]:
print(f"Registros: {df.shape[0]}")
print(f"Variables: {df.shape[1]}")
print("\nTipos de datos:")
print(df.dtypes)

Registros: 211043
Variables: 44

Tipos de datos:
src_ip                     object
src_port                    int64
dst_ip                     object
dst_port                    int64
proto                      object
service                    object
duration                  float64
src_bytes                   int64
dst_bytes                   int64
conn_state                 object
missed_bytes                int64
src_pkts                    int64
src_ip_bytes                int64
dst_pkts                    int64
dst_ip_bytes                int64
dns_query                  object
dns_qclass                  int64
dns_qtype                   int64
dns_rcode                   int64
dns_AA                     object
dns_RD                     object
dns_RA                     object
dns_rejected               object
ssl_version                object
ssl_cipher                 object
ssl_resumed                object
ssl_established            object
ssl_subject                object

1.2 Verificación y corrección de tipos de datos

Aunque `df.info()` no reporta valores nulos, algunas columnas numéricas
representan códigos o categorías (puertos, clases DNS, etiquetas), por lo que
se convierten a `object` para reflejar correctamente su naturaleza categórica
y evitar que sean tratadas como variables continuas en el análisis posterior.

In [3]:
df = df.astype({
    'src_port'   : 'object',
    'dst_port'   : 'object',
    'dns_qclass' : 'object',
    'dns_qtype'  : 'object',
    'dns_rcode'  : 'object',
    'label'      : 'object'
})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211043 entries, 0 to 211042
Data columns (total 44 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   src_ip                  211043 non-null  object 
 1   src_port                211043 non-null  object 
 2   dst_ip                  211043 non-null  object 
 3   dst_port                211043 non-null  object 
 4   proto                   211043 non-null  object 
 5   service                 211043 non-null  object 
 6   duration                211043 non-null  float64
 7   src_bytes               211043 non-null  int64  
 8   dst_bytes               211043 non-null  int64  
 9   conn_state              211043 non-null  object 
 10  missed_bytes            211043 non-null  int64  
 11  src_pkts                211043 non-null  int64  
 12  src_ip_bytes            211043 non-null  int64  
 13  dst_pkts                211043 non-null  int64  
 14  dst_ip_bytes        

### Estadísticas descriptivas

Se reportan las estadísticas descriptivas de las variables numéricas y categóricas
para entender la distribución, el rango de valores y la frecuencia de cada variable.

In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
src_port,211043.0,38646.519543,1.930727e+04,1.0,34608.0,44754.00000,51133.000000,6.552800e+04
dst_port,211043.0,3495.153770,1.019162e+04,0.0,65.0,80.00000,1253.000000,6.546700e+04
duration,211043.0,7.700887,5.641419e+02,0.0,0.0,0.00017,0.054196,9.351693e+04
src_bytes,211043.0,258113.564274,1.709490e+07,0.0,0.0,0.00000,130.000000,3.890855e+09
dst_bytes,211043.0,258804.571575,1.802563e+07,0.0,0.0,0.00000,89.000000,3.913853e+09
missed_bytes,211043.0,34432.344295,5.261621e+06,0.0,0.0,0.00000,0.000000,1.854527e+09
src_pkts,211043.0,9.595220,9.177882e+01,0.0,1.0,1.00000,4.000000,2.462300e+04
src_ip_bytes,211043.0,776.082206,2.229703e+04,0.0,48.0,82.00000,415.000000,6.522626e+06
dst_pkts,211043.0,3.846861,3.307058e+02,0.0,0.0,1.00000,2.000000,1.219420e+05
dst_ip_bytes,211043.0,1584.686628,1.901795e+05,0.0,0.0,40.00000,134.000000,8.639552e+07


Las variables numéricas presentan distribuciones muy sesgadas: `src_bytes`,
`dst_bytes`, `missed_bytes` y sus derivados tienen media muy superior a la mediana
(50%), lo que sugiere la presencia de valores atípicos. `duration` y los contadores
de paquetes muestran el mismo comportamiento. Esto indica que la mayoría de estas
variables requerirán transformaciones antes de entrenar el modelo.

In [10]:
df.describe(include='object').T

,count,unique,top,freq
src_ip,211043,51,192.168.1.30,61633
dst_ip,211043,753,192.168.1.190,47795
proto,211043,3,tcp,168747
service,211043,9,-,132032
conn_state,211043,13,S0,51937
dns_query,211043,726,-,176198
dns_AA,211043,3,-,176030
dns_RD,211043,3,-,176030
dns_RA,211043,3,-,176030
dns_rejected,211043,3,-,176030


Las variables categóricas muestran que `proto` tiene solo 3 protocolos únicos
(predomina `tcp`), `conn_state` tiene 13 estados posibles y `service` tiene 9.
Las columnas `src_ip` y `dst_ip` tienen 51 y 753 valores únicos respectivamente,
lo que confirma que son identificadores y no deben entrar en la matriz de características.
La variable objetivo `type` tiene 10 clases, siendo `normal` la más frecuente con
50.000 registros. Las columnas SSL, HTTP y DNS muestran valores `-` dominantes,
lo que indica que la mayoría de los flujos no usan esos protocolos.

### Datos duplicados

Se identifican y eliminan los registros duplicados, ya que filas idénticas
en todas sus columnas no aportan información adicional al modelo y pueden
sesgar el entrenamiento.

In [5]:
print('Número de registros duplicados: ', df.duplicated().sum())
df = df.drop_duplicates()
print('Registros después de eliminar duplicados: ', df.shape[0])

Número de registros duplicados:  20569
Registros después de eliminar duplicados:  190474


Se encontraron 20.569 registros duplicados (9.7% del total), los cuales fueron
eliminados. El dataframe quedó con 190.474 registros.

### Valores nulos

Se verifican los valores nulos en el dataframe. El parámetro `na_values=[' ']`
en la carga inicial garantiza que los espacios en blanco sean reconocidos
como nulos por pandas

In [6]:
df.isnull().sum()

src_ip                    0
src_port                  0
dst_ip                    0
dst_port                  0
proto                     0
service                   0
duration                  0
src_bytes                 0
dst_bytes                 0
conn_state                0
missed_bytes              0
src_pkts                  0
src_ip_bytes              0
dst_pkts                  0
dst_ip_bytes              0
dns_query                 0
dns_qclass                0
dns_qtype                 0
dns_rcode                 0
dns_AA                    0
dns_RD                    0
dns_RA                    0
dns_rejected              0
ssl_version               0
ssl_cipher                0
ssl_resumed               0
ssl_established           0
ssl_subject               0
ssl_issuer                0
http_trans_depth          0
http_method               0
http_uri                  0
http_version              0
http_request_body_len     0
http_response_body_len    0
http_status_code    

El dataframe no presenta valores nulos en ninguna de sus 44 columnas,
por lo que no se requiere ningún tratamiento adicional para datos faltantes.

### Hallazgos 

**Hallazgos:**
- El dataframe tiene 211.043 registros y 44 variables. Tras la limpieza inicial
  quedaron 190.474 registros.
- `src_ip` y `dst_ip` son identificadores de red con 51 y 753 valores únicos
  respectivamente, por lo que no deben entrar en la matriz de características.
- Las variables `dns_qclass`, `dns_qtype`, `dns_rcode`, `src_port`, `dst_port`
  y `label` estaban almacenadas como `int64` pero representan códigos o categorías.
- Se encontraron 20.569 registros duplicados (9.7% del total).
- No se encontraron valores nulos en ninguna columna.
- Las variables numéricas como `src_bytes`, `dst_bytes`, `duration` y sus derivados
  presentan distribuciones muy sesgadas con valores atípicos.
- La variable objetivo es `type`, con 10 clases siendo `normal` la más frecuente.

**Soluciones:**
- Corregir los tipos de dato de las columnas categóricas almacenadas como numéricas.
- Eliminar los 20.569 registros duplicados.
- Eliminar `src_ip` y `dst_ip` por ser identificadores sin valor predictivo.

### Descarte de variables irrelevantes

Se eliminan `src_ip` y `dst_ip` por ser identificadores de red sin valor
predictivo, y `type` por ser redundante con la variable objetivo `label`.

In [7]:
df = df.drop(columns=['src_ip', 'dst_ip', 'type'])
df.head()

,src_port,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_request_body_len,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label
0,4444,49178,tcp,-,290.371539,101568,2592,OTH,0,108,...,0,0,0,-,-,-,-,-,-,1
1,49180,8080,tcp,-,0.000102,0,0,REJ,0,1,...,0,0,0,-,-,-,-,-,-,1
2,49180,8080,tcp,-,0.000148,0,0,REJ,0,1,...,0,0,0,-,-,-,-,-,-,1
3,49180,8080,tcp,-,0.000113,0,0,REJ,0,1,...,0,0,0,-,-,-,-,-,-,1
4,49180,8080,tcp,-,0.000130,0,0,REJ,0,1,...,0,0,0,-,-,-,-,-,-,1


El dataframe quedó con 41 variables tras eliminar las 3 columnas irrelevantes.

## 2. Análisis exploratorio de variables categóricas

### Identificación de variables nominales y ordinales

Se listan las variables categóricas del dataframe y se clasifican según
si sus categorías tienen orden inherente (ordinales) o no (nominales).